In [1]:
pip install pandas numpy faker openpyxl

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
from faker import Faker
import random
from datetime import date
import os

fake = Faker()
np.random.seed(42)
random.seed(42)

In [4]:
regions     = ['North', 'South', 'East', 'West', 'Central']
categories  = ['Electronics', 'Apparel', 'Food & Beverage', 'Home & Living', 'Sports']
products    = {
    'Electronics':     ['Laptop', 'Tablet', 'Headphones', 'Smartwatch', 'Keyboard'],
    'Apparel':         ['Jacket', 'Sneakers', 'T-Shirt', 'Jeans', 'Cap'],
    'Food & Beverage': ['Coffee Beans', 'Protein Bar', 'Green Tea', 'Olive Oil', 'Pasta'],
    'Home & Living':   ['Lamp', 'Cushion', 'Rug', 'Candle', 'Plant Pot'],
    'Sports':          ['Yoga Mat', 'Dumbbell', 'Running Shoes', 'Water Bottle', 'Resistance Band'],
}
salesperson = [fake.name() for _ in range(20)]

In [5]:
rows = []
dates = pd.date_range(start='2023-01-01', end='2024-12-31', freq='D')

for _ in range(2400):
    txn_date  = random.choice(dates)
    category  = random.choice(categories)
    product   = random.choice(products[category])
    region    = random.choice(regions)
    sales_rep = random.choice(salesperson)
    quantity  = random.randint(1, 50)

    unit_price  = round(random.uniform(5, 500), 2)
    unit_cost   = round(unit_price * random.uniform(0.45, 0.75), 2)
    budget      = round(unit_price * quantity * random.uniform(0.80, 1.20), 2)

    sales_amt   = round(unit_price * quantity, 2)
    cost_amt    = round(unit_cost  * quantity, 2)
    profit      = round(sales_amt - cost_amt, 2)

    rows.append({
        'transaction_id': f'TXN{_ + 1:05d}',
        'date':           txn_date.date(),
        'year':           txn_date.year,
        'month':          txn_date.month,
        'quarter':        f'Q{txn_date.quarter}',
        'region':         region,
        'category':       category,
        'product':        product,
        'sales_rep':      sales_rep,
        'quantity':       quantity,
        'unit_price':     unit_price,
        'unit_cost':      unit_cost,
        'sales_amount':   sales_amt,
        'cost_amount':    cost_amt,
        'profit':         profit,
        'budget_amount':  budget,
    })

df = pd.DataFrame(rows)
print(f"Shape: {df.shape}")
df.head()

Shape: (2400, 16)


,transaction_id,date,year,month,quarter,region,category,product,sales_rep,quantity,unit_price,unit_cost,sales_amount,cost_amount,profit,budget_amount
0,TXN00001,2024-10-16,2024,10,Q4,East,Electronics,Laptop,Ryan Richards,15,74.07,35.61,1111.05,534.15,576.90,1218.01
1,TXN00002,2024-07-12,2024,7,Q3,West,Electronics,Keyboard,Natalie Farley,2,51.38,26.71,102.76,53.42,49.34,106.95
2,TXN00003,2024-07-28,2024,7,Q3,West,Apparel,Cap,Ryan Richards,29,296.69,205.56,8604.01,5961.24,2642.77,6905.57
3,TXN00004,2023-06-13,2023,6,Q2,East,Home & Living,Rug,Calvin Williams,14,478.82,263.82,6703.48,3693.48,3010.00,5611.47
4,TXN00005,2023-04-10,2023,4,Q2,Central,Food & Beverage,Green Tea,Steven Lewis,3,366.22,223.71,1098.66,671.13,427.53,1306.58


In [6]:
# --- inject 3% nulls to simulate real-world data quality issues ---
for col in ['sales_rep', 'region', 'quantity']:
    df.loc[df.sample(frac=0.03).index, col] = np.nan

print("Nulls before cleaning:")
print(df.isnull().sum())

# --- clean ---
df['sales_rep'] = df['sales_rep'].fillna('Unknown')
df['region']    = df['region'].fillna(df['region'].mode()[0])
df['quantity']  = df['quantity'].fillna(df['quantity'].median()).astype(int)

# --- fix data types ---
df['date']  = pd.to_datetime(df['date'])
df['year']  = df['year'].astype(int)
df['month'] = df['month'].astype(int)

# --- add variance column (actual vs budget) ---
df['budget_variance']    = round(df['sales_amount'] - df['budget_amount'], 2)
df['variance_pct']       = round((df['budget_variance'] / df['budget_amount']) * 100, 2)
df['gross_margin_pct']   = round((df['profit'] / df['sales_amount']) * 100, 2)

print("\nNulls after cleaning:")
print(df.isnull().sum())
print(f"\nFinal shape: {df.shape}")
df.dtypes

Nulls before cleaning:
transaction_id     0
date               0
year               0
month              0
quarter            0
region            72
category           0
product            0
sales_rep         72
quantity          72
unit_price         0
unit_cost          0
sales_amount       0
cost_amount        0
profit             0
budget_amount      0
dtype: int64

Nulls after cleaning:
transaction_id      0
date                0
year                0
month               0
quarter             0
region              0
category            0
product             0
sales_rep           0
quantity            0
unit_price          0
unit_cost           0
sales_amount        0
cost_amount         0
profit              0
budget_amount       0
budget_variance     0
variance_pct        0
gross_margin_pct    0
dtype: int64

Final shape: (2400, 19)


transaction_id              object
date                datetime64[ns]
year                         int64
month                        int64
quarter                     object
region                      object
category                    object
product                     object
sales_rep                   object
quantity                     int64
unit_price                 float64
unit_cost                  float64
sales_amount               float64
cost_amount                float64
profit                     float64
budget_amount              float64
budget_variance            float64
variance_pct               float64
gross_margin_pct           float64
dtype: object

In [7]:
print("=== Summary Stats ===")
print(f"Total Revenue :  €{df['sales_amount'].sum():,.0f}")
print(f"Total Profit  :  €{df['profit'].sum():,.0f}")
print(f"Avg Margin %  :  {df['gross_margin_pct'].mean():.1f}%")
print(f"Date Range    :  {df['date'].min().date()} → {df['date'].max().date()}")
print(f"Unique Products: {df['product'].nunique()}")
print()
print(df.groupby('region')['sales_amount'].sum().sort_values(ascending=False))

=== Summary Stats ===
Total Revenue :  €15,348,110
Total Profit  :  €6,140,641
Avg Margin %  :  40.0%
Date Range    :  2023-01-01 → 2024-12-31
Unique Products: 25

region
West       3706299.76
East       3186358.45
South      3007219.07
North      2757408.93
Central    2690823.40
Name: sales_amount, dtype: float64


In [8]:
output_path = os.path.join(os.path.expanduser('~'), 'Desktop', 'retail_sales_clean.csv')
df.to_csv(output_path, index=False)
print(f"Saved: {output_path}")
print(f"Rows: {len(df):,}  |  Columns: {len(df.columns)}")

Saved: C:\Users\SREE HARI\Desktop\retail_sales_clean.csv
Rows: 2,400  |  Columns: 19


In [9]:
import mysql.connector
import pandas as pd
import os
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment

conn = mysql.connector.connect(
    host='127.0.0.1',
    port=3306,
    user='root',
    password='Root@1234',
    database='retail_db'
)

queries = {
    'Revenue by Region': """
        SELECT region,
               ROUND(SUM(sales_amount),2)     AS total_revenue,
               ROUND(SUM(profit),2)           AS total_profit,
               ROUND(AVG(gross_margin_pct),2) AS avg_margin_pct
        FROM retail_sales
        GROUP BY region ORDER BY total_revenue DESC
    """,
    'Monthly Trend': """
        SELECT year, month,
               ROUND(SUM(sales_amount),2)    AS monthly_revenue,
               ROUND(SUM(profit),2)          AS monthly_profit,
               ROUND(SUM(budget_amount),2)   AS monthly_budget,
               ROUND(SUM(budget_variance),2) AS monthly_variance
        FROM retail_sales
        GROUP BY year, month ORDER BY year, month
    """,
    'Top 10 Products': """
        SELECT product, category,
               ROUND(SUM(sales_amount),2) AS total_revenue,
               ROUND(SUM(profit),2)       AS total_profit,
               SUM(quantity)              AS units_sold
        FROM retail_sales
        GROUP BY product, category
        ORDER BY total_revenue DESC LIMIT 10
    """,
    'Margin by Category': """
        SELECT category,
               ROUND(SUM(sales_amount),2)     AS total_revenue,
               ROUND(SUM(cost_amount),2)       AS total_cost,
               ROUND(SUM(profit),2)            AS total_profit,
               ROUND(AVG(gross_margin_pct),2)  AS avg_margin_pct
        FROM retail_sales
        GROUP BY category ORDER BY avg_margin_pct DESC
    """,
    'Budget Variance': """
        SELECT year, quarter,
               ROUND(SUM(sales_amount),2)    AS actual_revenue,
               ROUND(SUM(budget_amount),2)   AS budget_revenue,
               ROUND(SUM(budget_variance),2) AS total_variance,
               ROUND(AVG(variance_pct),2)    AS avg_variance_pct
        FROM retail_sales
        GROUP BY year, quarter ORDER BY year, quarter
    """
}

output_path = os.path.join(os.path.expanduser('~'), 'Desktop', 'retail_kpi_report.xlsx')

with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    for sheet_name, query in queries.items():
        df = pd.read_sql(query, conn)
        df.to_excel(writer, sheet_name=sheet_name, index=False)

        ws = writer.sheets[sheet_name]
        header_fill = PatternFill(fill_type='solid', fgColor='1F4E79')
        header_font = Font(bold=True, color='FFFFFF')

        for cell in ws[1]:
            cell.fill = header_fill
            cell.font = header_font
            cell.alignment = Alignment(horizontal='center')

        for col in ws.columns:
            max_len = max(len(str(cell.value or '')) for cell in col) + 4
            ws.column_dimensions[col[0].column_letter].width = max_len

conn.close()
print(f"KPI report saved to Desktop: retail_kpi_report.xlsx")
print(f"Sheets created: {list(queries.keys())}")

C:\Users\SREE HARI\AppData\Local\Temp\ipykernel_49040\3584025338.py:66: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


KPI report saved to Desktop: retail_kpi_report.xlsx
Sheets created: ['Revenue by Region', 'Monthly Trend', 'Top 10 Products', 'Margin by Category', 'Budget Variance']
